Загрузка Robometr

In [1]:

from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="aliangdw/Robometer-4B-LIBERO",
    local_dir="../models/Robometer-4B-LIBERO",
)

print("Downloaded to:", path)


/home/msi/projects/TrainerVLA/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 11 files: 100%|██████████| 11/11 [1:22:04<00:00, 447.67s/it]

Downloaded to: /home/msi/projects/TrainerVLA/models/Robometer-4B-LIBERO


Столкнулся с ошибкой при запуске lerobot-train на sarm:

Необходимо передать для запуска: reward_model.state_key=observation.state

Ничего проще как исправить этот кусоки просто локально поправить у себя venv (((

(Это плохо для воспроизодимости, я понимаю, и к тому же придётся делать ручками )

в файле `.venv/lib/python3.12/site-packages/lerobot/rewards/sarm/processor_sarm.py`

После этого куска

```python
state_key = self.config.state_key
state_data = observation.get(state_key)

if isinstance(state_data, torch.Tensor):
    state_tensor = state_data.float()
else:
    state_tensor = torch.tensor(state_data, dtype=torch.float32)
```
Добавить этот кусок:
```python
# ZERO-STATE ABLATION:
# Do not allow SARM to use robot state information.
state_tensor = torch.zeros_like(state_tensor)

if state_tensor.ndim == 2:
    state_tensor = state_tensor.unsqueeze(0)
elif state_tensor.ndim == 1:
    state_tensor = state_tensor.unsqueeze(0).unsqueeze(0)
```

Проблемы при запуске robometr:

`/robometer/robometer/utils/setup_utils.py`

После этого куска
```python
for ckpt_key, ckpt_value in checkpoint_state_dict.items():
    if ckpt_key in model_keys:
        # Direct match - use as is
        remapped_state_dict[ckpt_key] = ckpt_value
        direct_match_count += 1
    else:
        # Try different remapping strategies
        potential_keys = []
```
Добавить этот кусок:
```python
    # ZERO-STATE ABLATION:
    # Do not allow SARM to use robot state information.
    state_tensor = torch.zeros_like(state_tensor)

    if state_tensor.ndim == 2:
        state_tensor = state_tensor.unsqueeze(0)
    elif state_tensor.ndim == 1:
        state_tensor = state_tensor.unsqueeze(0).unsqueeze(0)
```


`robometer/robometer/evals/eval_server.py`

Исправить это
```python
model_output, extra = model(
    input_ids=batch_inputs["input_ids"],
    attention_mask=batch_inputs["attention_mask"],
    pixel_values=batch_inputs.get("pixel_values", None),
    pixel_values_videos=batch_inputs.get("pixel_values_videos", None),
    image_grid_thw=batch_inputs.get("image_grid_thw", None),
    video_grid_thw=batch_inputs.get("video_grid_thw", None),
    second_per_grid_ts=batch_inputs.get("second_per_grid_ts", None),
    sample_type=sample_type,
    timing_raw=None,
)
```
На это:
```python
model_output, extra = model(
    input_ids=batch_inputs["input_ids"],
    attention_mask=batch_inputs["attention_mask"],
    pixel_values=batch_inputs.get("pixel_values", None),
    pixel_values_videos=batch_inputs.get("pixel_values_videos", None),
    image_grid_thw=batch_inputs.get("image_grid_thw", None),
    video_grid_thw=batch_inputs.get("video_grid_thw", None),

    # Required by Qwen3-VL / transformers 5.5+
    mm_token_type_ids=batch_inputs.get("mm_token_type_ids", None),

    second_per_grid_ts=batch_inputs.get("second_per_grid_ts", None),
    sample_type=sample_type,
    timing_raw=None,
)
```

`.venv/lib/python3.12/site-packages/lerobot/envs/libero.py`

Исправить это
```python
class LiberoEnv(gym.Env):
    metadata = {"render_modes": ["rgb_array"], "render_fps": 80}
```
На это:
```python
class LiberoEnv(gym.Env):
    metadata = {"render_modes": ["rgb_array"], "render_fps": 20}
```

## Результаты и выводы

In [45]:
from pathlib import Path
import numpy as np
import json

log_dir = Path('../logs_task_4')
exps = dict()
seeds = ["s_1001", "s_1002"]
for exp in log_dir.iterdir():
    exp_name = exp.name 
    t_id, d_cnt = exp_name.split("_")[-2:]
    t_id = int(t_id[1:])
    d_cnt = int(d_cnt[1:])
    exps[exp_name] = {
        "task_id": t_id,
        "demos": d_cnt,
        "sarm": [],
        "robometer": [],
        "success_rate": 0
    }


    for exp_s in seeds:
        
        with open(exp / exp_s / "eval_info.json", 'r', encoding='utf-8') as file:
            successes_rate_seed = json.load(file)["overall"]["avg_sum_reward"]

        exps[exp_name]["success_rate"] += successes_rate_seed


        exp_res_dir = exp / exp_s / "videos" / f"libero_goal_{t_id}"
        for exp_res_path in  sorted(exp_res_dir.iterdir()):
            if exp_res_path.name.endswith("_sarm_rewards.npy"):
                exps[exp_name]['sarm'].append(exp_res_path)
            elif exp_res_path.name.endswith("_rewards.npy"):
                exps[exp_name]['robometer'].append(exp_res_path)
                
    exps[exp_name]["success_rate"] /= len(seeds)
    

In [51]:
def sarm_rewards_to_success(rewards, n_points=10):
    return rewards[-n_points:].mean() 

def robometer_rewards_to_success(rewards):
    return rewards[-1]

for exp_name, exp in exps.items():
    for method in ["sarm", "robometer"]:

        episode_scores = []
        for path in exp[method]:
            rewards = np.load(path)

            if method == "sarm":
                score = sarm_rewards_to_success(rewards)
            elif method == "robometer":
                score = robometer_rewards_to_success(rewards)

            episode_scores.append(score)

        checkpoint_score = np.mean(episode_scores)        
        if method == "sarm":
            exp["sarm_score"] = checkpoint_score
        elif method == "robometer":
            exp["robometer_score"] = checkpoint_score


In [52]:
import pandas as pd 

df = pd.DataFrame.from_dict(exps, orient="index", columns=["task_id", "demos", "success_rate", "sarm_score", "robometer_score"])
df.sort_values(by=['task_id', 'demos'])

,task_id,demos,success_rate,sarm_score,robometer_score
smolvla_libero90_t0_d0,0,0,0.00,0.579099,0.326236
smolvla_libero90_t0_d5,0,5,0.70,0.718314,0.642690
smolvla_libero90_t0_d10,0,10,1.00,0.718152,0.735732
smolvla_libero90_t0_d25,0,25,1.00,0.693766,0.734640
smolvla_libero90_t1_d0,1,0,0.00,0.558466,0.383212
smolvla_libero90_t1_d5,1,5,0.95,0.570869,0.610849
smolvla_libero90_t1_d10,1,10,0.85,0.532480,0.580584
smolvla_libero90_t1_d25,1,25,0.95,0.576297,0.607428
smolvla_libero90_t2_d0,2,0,0.00,0.519579,0.391775
smolvla_libero90_t2_d5,2,5,0.85,0.503930,0.719690


In [54]:
from scipy.stats import spearmanr, kendalltau
import pandas as pd

ranking_results = []

for task_id, task_df in df.groupby("task_id"):

    true_success = task_df["success_rate"].to_numpy()
    sarm_score = task_df["sarm_score"].to_numpy()
    robo_score = task_df["robometer_score"].to_numpy()

    sarm_spearman, _ = spearmanr(true_success, sarm_score)
    robo_spearman, _ = spearmanr(true_success, robo_score)

    sarm_kendall, _ = kendalltau(true_success, sarm_score)
    robo_kendall, _ = kendalltau(true_success, robo_score)

    ranking_results.append({
        "task_id": task_id,
        "sarm_spearman": sarm_spearman,
        "robometer_spearman": robo_spearman,
        "sarm_kendall": sarm_kendall,
        "robometer_kendall": robo_kendall,
    })

ranking_df = pd.DataFrame(ranking_results)

ranking_df

,task_id,sarm_spearman,robometer_spearman,sarm_kendall,robometer_kendall
0,0,0.316228,0.948683,0.182574,0.912871
1,1,0.737865,0.948683,0.547723,0.912871
2,2,0.737865,0.948683,0.547723,0.912871


Видно, что ранжирует лучше robometer, но это и неудивительно ведь я взял готовую Robometer-4B-LIBERO, которая обучалась в том числе на задаче libero_goal

Прямая оптимизация политики по SARM или Robometer может привести к reward hacking: политика научится находить состояния, которые критик оценивает как высокий progress, даже если реальная задача не выполнена. Это особенно вероятно из-за ошибок критика и distribution shift. Дополнительно можно провести более безопасный эксперимент с RA-BC, где learned progress используется только для взвешивания примеров в behavior cloning.
